# 02 · The All-Seeing Eye – token usage per user
Five heroes generate traffic. The gateway's `llm-emit-token-metric` policy tags every request with **User / Model / Tier**, and the APIM **LLM gateway logs** record exact prompt/completion tokens per request.
Both end up in Log Analytics (ingestion takes a few minutes).

In [ ]:
import sys; sys.path.insert(0, '.')
from heimdal_helpers import *

import random
prompts = ["Give me a one-line Avengers fun fact.", "Summarise the plot of Infinity War in 2 sentences.",
           "What is the capital of Asgard?", "List three Iron Man suits."]
rows = []
for user in USERS:
    for _ in range(random.randint(2, 4)):
        p = random.choice(prompts)
        r = ask(p, user=user)
        rows.append({'prompt': p[:40], **r})
show(rows, cols=('user', 'tier', 'model', 'prompt_tokens', 'completion_tokens'))

### Per-user usage from custom metrics (Application Insights → `AppMetrics`)

In [ ]:
import sys; sys.path.insert(0, '.')
from heimdal_helpers import *

df = kql(KQL_TOKENS_BY_USER_METRIC, hours=3)
df

### Per-user usage from LLM gateway logs (exact per-request tokens + model)

In [ ]:
import sys; sys.path.insert(0, '.')
from heimdal_helpers import *

usage = kql(KQL_TOKENS_BY_USER_LLMLOG, hours=3)
display(usage)
usage.groupby('User')['Total'].sum().sort_values().plot.barh(title='Total tokens by user')

> If tables are empty, wait 3–5 minutes and re-run. Custom metrics also appear in *Application Insights → Metrics* (namespace `heimdal`).